# Task 2: Loading and Preparing Combined Data
## Overview
This section loads the data for Task 2, creates the combined mutation and methylation features, and prepares them for classification.

In [13]:
# Install required libraries in Google Colab
!pip install pandas numpy scikit-learn xgboost cudf-cu12 --extra-index-url=https://pypi.nvidia.com

# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.metrics import f1_score, precision_score, recall_score
from xgboost import XGBClassifier
import cudf
import os
import torch

# Check GPU availability
if torch.cuda.is_available():
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not available. Please enable GPU runtime in Colab (Runtime > Change runtime type > GPU).")
    raise SystemExit

# Set working directory
os.chdir('/content/work')

Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com
GPU is available: Tesla T4


## Phase 1: Loading Raw Data for Task 2
### Reading and Understanding the Data
- We load the training and testing data using `cudf.read_csv` (accelerated with GPU) and convert to Pandas DataFrames.
- Files loaded: `train_muts_data.csv`, `test_muts_data.csv`, `train_feats.csv`, `test_feats.csv`, `train_meth_data.csv`, `test_meth_data.csv`, `100_genes.csv`.

In [14]:
# Load data for Task 2
print("Loading data for Task 2...")
try:
    train_muts = cudf.read_csv('train_muts_data.csv').to_pandas()
    test_muts = cudf.read_csv('test_muts_data.csv').to_pandas()
    train_feats = cudf.read_csv('train_feats.csv').to_pandas()
    test_feats = cudf.read_csv('test_feats.csv').to_pandas()
    train_meth = cudf.read_csv('train_meth_data.csv').to_pandas()
    test_meth = cudf.read_csv('test_meth_data.csv').to_pandas()
    genes_100 = cudf.read_csv('100_genes.csv').to_pandas()
except FileNotFoundError as e:
    print(f"Error: File not found. Ensure all CSV files are uploaded to Colab: {e}")
    raise SystemExit

# Check for missing values
print("Checking for missing values in Task 2 data...")
for df, name in [(train_muts, 'train_muts'), (test_muts, 'test_muts'), (train_feats, 'train_feats'),
                 (test_feats, 'test_feats'), (train_meth, 'train_meth'), (test_meth, 'test_meth')]:
    print(f"{name}: {df.isnull().sum().sum()} missing values")

Loading data for Task 2...
Checking for missing values in Task 2 data...
train_muts: 0 missing values
test_muts: 0 missing values
train_feats: 0 missing values
test_feats: 0 missing values
train_meth: 0 missing values
test_meth: 0 missing values


## Phase 2: Feature Engineering for Task 2 (Mutations)
### Creating Mutation Features
- We create mutation features from `train_muts` and `test_muts`.
- Features include: total mutations per patient, mutations per variant classification, mutations per gene and variant, mutation types, and normalized mutation rates.

In [ ]:
# Feature Engineering for Task 2 (Mutations)
print("Creating mutation features for Task 2...")
train_total_muts = train_muts.groupby('case_id').size().reset_index(name='Total_Mutations')
test_total_muts = test_muts.groupby('case_id').size().reset_index(name='Total_Mutations')
train_var_counts = train_muts.groupby(['case_id', 'Variant_Classification']).size().unstack(fill_value=0)
train_var_counts.columns = [f'Mutations_{col}' for col in train_var_counts.columns]
test_var_counts = test_muts.groupby(['case_id', 'Variant_Classification']).size().unstack(fill_value=0)
test_var_counts.columns = [f'Mutations_{col}' for col in test_var_counts.columns]
train_gene_var = train_muts.groupby(['case_id', 'Gene_name', 'Variant_Classification']).size().unstack(level=[1, 2], fill_value=0)
train_gene_var.columns = [f'Mutations_in_{gene}_{var}' for gene, var in train_gene_var.columns]
test_gene_var = test_muts.groupby(['case_id', 'Gene_name', 'Variant_Classification']).size().unstack(level=[1, 2], fill_value=0)
test_gene_var.columns = [f'Mutations_in_{gene}_{var}' for gene, var in test_gene_var.columns]
def classify_mutation(row):
    ref, tumor = row['Reference_Allele'], row['Tumor_Seq_Allele1']
    if len(ref) == len(tumor) == 1:
        transitions = {('A', 'G'), ('G', 'A'), ('C', 'T'), ('T', 'C')}
        return 'Transition' if (ref, tumor) in transitions else 'Transversion'
    elif len(ref) > len(tumor):
        return 'Deletion'
    elif len(ref) < len(tumor):
        return 'Insertion'
    return 'Other'
train_muts['Mut_Type'] = train_muts.apply(classify_mutation, axis=1)
test_muts['Mut_Type'] = test_muts.apply(classify_mutation, axis=1)
train_mut_types = train_muts.groupby(['case_id', 'Mut_Type']).size().unstack(fill_value=0)
train_mut_types.columns = [f'Mutations_{col}' for col in train_mut_types.columns]
test_mut_types = test_muts.groupby(['case_id', 'Mut_Type']).size().unstack(fill_value=0) 
test_mut_types.columns = [f'Mutations_{col}' for col in test_mut_types.columns]

# Normalized mutation rate by gene length
genes_100['Length'] = genes_100['Sequence'].str.len()
train_norm_muts = train_muts.groupby(['case_id', 'Gene_name']).size().reset_index(name='Count')
train_norm_muts = train_norm_muts.merge(genes_100[['gene', 'Length']], left_on='Gene_name', right_on='gene')
train_norm_muts['Norm_Mutations'] = train_norm_muts['Count'] / train_norm_muts['Length']
train_norm_muts = train_norm_muts.pivot(index='case_id', columns='Gene_name', values='Norm_Mutations').fillna(0)
train_norm_muts.columns = [f'Norm_Mutations_in_{col}' for col in train_norm_muts.columns]
test_norm_muts = test_muts.groupby(['case_id', 'Gene_name']).size().reset_index(name='Count')
test_norm_muts = test_norm_muts.merge(genes_100[['gene', 'Length']], left_on='Gene_name', right_on='gene')
test_norm_muts['Norm_Mutations'] = test_norm_muts['Count'] / test_norm_muts['Length']
test_norm_muts = test_norm_muts.pivot(index='case_id', columns='Gene_name', values='Norm_Mutations').fillna(0)
test_norm_muts.columns = [f'Norm_Mutations_in_{col}' for col in test_norm_muts.columns]
train_features = train_total_muts.set_index('case_id').join([train_var_counts, train_gene_var, train_mut_types,
                                                             train_norm_muts]).reset_index()
test_features = test_total_muts.set_index('case_id').join([test_var_counts, test_gene_var, test_mut_types,
                                                           test_norm_muts]).reset_index()

Creating mutation features for Task 2...


## Phase 3: Feature Engineering for Task 2 (Methylation)
### Creating Methylation Features
- We create features from methylation data (`train_meth`, `test_meth`) for Task 2.
- Features include: average methylation per gene, standard deviation, and proportion of high-methylation sites (beta_val > 0.7).
- We add an interaction feature (`Mut_Meth_Interaction`) to capture cases where mutations occur with high methylation.

In [17]:
# Methylation Features
print("Creating methylation features for Task 2...")
train_meth_avg = train_meth.groupby(['case_id', 'matching_genes'])['beta_val'].mean().unstack(fill_value=0)
train_meth_avg.columns = [f'Meth_Avg_{col}' for col in train_meth_avg.columns]
test_meth_avg = test_meth.groupby(['case_id', 'matching_genes'])['beta_val'].mean().unstack(fill_value=0)
test_meth_avg.columns = [f'Meth_Avg_{col}' for col in test_meth_avg.columns]
train_meth_std = train_meth.groupby(['case_id', 'matching_genes'])['beta_val'].std().unstack(fill_value=0)
train_meth_std.columns = [f'Meth_Std_{col}' for col in train_meth_std.columns]
test_meth_std = test_meth.groupby(['case_id', 'matching_genes'])['beta_val'].std().unstack(fill_value=0)
test_meth_std.columns = [f'Meth_Std_{col}' for col in test_meth_std.columns]
train_meth_high = train_meth[train_meth['beta_val'] > 0.7].groupby(['case_id', 'matching_genes']).size().unstack(fill_value=0)
train_meth_high = train_meth_high.div(train_meth.groupby(['case_id', 'matching_genes']).size().unstack(fill_value=1)).fillna(0)
train_meth_high.columns = [f'Meth_High_Prop_{col}' for col in train_meth_high.columns]
test_meth_high = test_meth[test_meth['beta_val'] > 0.7].groupby(['case_id', 'matching_genes']).size().unstack(fill_value=0)
test_meth_high = test_meth_high.div(test_meth.groupby(['case_id', 'matching_genes']).size().unstack(fill_value=1)).fillna(0)
test_meth_high.columns = [f'Meth_High_Prop_{col}' for col in test_meth_high.columns]
train_interaction = train_muts.merge(train_meth[train_meth['beta_val'] > 0.7], left_on=['case_id', 'Gene_name'],
                                     right_on=['case_id', 'matching_genes'], how='inner')
train_interaction = train_interaction.groupby('case_id').size().reset_index(name='Mut_Meth_Interaction')
test_interaction = test_muts.merge(test_meth[test_meth['beta_val'] > 0.7], left_on=['case_id', 'Gene_name'],
                                   right_on=['case_id', 'matching_genes'], how='inner')
test_interaction = test_interaction.groupby('case_id').size().reset_index(name='Mut_Meth_Interaction')
train_interaction = train_interaction.set_index('case_id').reindex(train_features['case_id']).fillna(0).reset_index()
test_interaction = test_interaction.set_index('case_id').reindex(test_features['case_id']).fillna(0).reset_index()

# Combine mutation and methylation features
train_combined = train_features.set_index('case_id').join([train_meth_avg, train_meth_std, train_meth_high,
                                                           train_interaction.set_index('case_id')]).reset_index()
test_combined = test_features.set_index('case_id').join([test_meth_avg, test_meth_std, test_meth_high,
                                                         test_interaction.set_index('case_id')]).reset_index()

# Optionally save the combined data as CSV for future use
train_combined.to_csv('train_combined.csv', index=False)
test_combined.to_csv('test_combined.csv', index=False)
print("Combined features saved as 'train_combined.csv' and 'test_combined.csv'.")

Creating methylation features for Task 2...
Combined features saved as 'train_combined.csv' and 'test_combined.csv'.


## Phase 4: Building and Evaluating Classifier for Task 2
### Training and Evaluating Classifiers
- We preprocess the combined features by removing constant features and selecting the top 100 features.
- We train two classifiers (RandomForest and XGBoost) using `GridSearchCV` and evaluate them.
- Predictions are saved to `task2_predictions.csv`.

In [ ]:
# Prepare y_train from training labels
y_train = train_feats['Label'].astype(int) - 1  # Convert labels to 0/1 (assuming 1,2 to 0,1)

# Classifier for Task 2
print("Training Task 2 classifier...")
X_train_combined = train_combined.drop(columns=['case_id']).fillna(0)
X_test_combined = test_combined.drop(columns=['case_id']).fillna(0)
X_test_combined = X_test_combined.reindex(columns=X_train_combined.columns, fill_value=0)

variance_threshold = VarianceThreshold(threshold=0)
X_train_combined_var = variance_threshold.fit_transform(X_train_combined)
X_test_combined_var = variance_threshold.transform(X_test_combined)

selector = SelectKBest(score_func=f_classif, k=100)
X_train_combined_selected = selector.fit_transform(X_train_combined_var, y_train)
X_test_combined_selected = selector.transform(X_test_combined_var)

X_train_split, X_val, y_train_split, y_val = train_test_split(X_train_combined_selected, y_train, test_size=0.2, stratify=y_train, random_state=42)
classifiers = {
    'RandomForest': RandomForestClassifier(random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(device='cuda', tree_method='hist', random_state=42)
}
best_classifier = None
best_score = 0

for name, classifier in classifiers.items():
    if name == 'RandomForest':
        param_grid = {'n_estimators': [100, 200, 300], 'max_depth': [3, 5, 7], 'min_samples_split': [2, 5]}
    else:
        param_grid = {'max_depth': [3, 5, 7], 'learning_rate': [0.01, 0.1, 0.3], 'n_estimators': [100, 200, 300]}
    grid_search = GridSearchCV(classifier, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
    grid_search.fit(X_train_split, y_train_split)
    y_val_pred = grid_search.predict(X_val)
    error = (y_val_pred != y_val).mean()
    f1 = f1_score(y_val, y_val_pred, average='weighted')
    print(f"{name} - Validation error: {error}")
    print(f"{name} - F1-Score: {f1}")
    print(f"{name} - Precision: {precision_score(y_val, y_val_pred, average='weighted')}")
    print(f"{name} - Recall: {recall_score(y_val, y_val_pred, average='weighted')}")
    if f1 > best_score:
        best_score = f1
        best_classifier = grid_search.best_estimator_

# Save predictions
best_classifier.fit(X_train_combined_selected, y_train)
y_test_pred = best_classifier.predict(X_test_combined_selected) + 1
pd.DataFrame({'id_case': test_combined['case_id'], 'label_predict': y_test_pred}).to_csv('task2_predictions.csv', index=False)

# Generate and Save Confusion Matrix
print("Generating confusion matrix visualization...")
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

plt.figure(figsize=(8, 6))
disp = ConfusionMatrixDisplay.from_predictions(y_val, y_val_pred,
                                             labels=[0, 1],
                                             display_labels=['HNSC', 'LUSC'],
                                             cmap='Blues')
plt.title('Task 2: Integrated Classification\nConfusion Matrix')
plt.tight_layout()
plt.savefig('out/confusion_matrix_task2.png', dpi=300, bbox_inches='tight')
plt.close()

print("Task 2 completed. Predictions and visualizations saved.")

Training Task 2 classifier...
RandomForest - Validation error: 0.42857142857142855
RandomForest - F1-Score: 0.565718987724395
RandomForest - Precision: 0.5730112181725086
RandomForest - Recall: 0.5714285714285714


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [21:17:51] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


XGBoost - Validation error: 0.422360248447205
XGBoost - F1-Score: 0.5755076129888723
XGBoost - Precision: 0.5778647943109191
XGBoost - Recall: 0.577639751552795
Task 2 completed. Predictions saved.


### Confusion Matrix Interpretation

The confusion matrix visualizes the performance improvement achieved by integrating mutation and methylation data:
- Rows represent actual classes (True labels)
- Columns represent predicted classes
- Diagonal elements show correct predictions
- Off-diagonal elements show misclassifications

Key observations:
1. Improved classification accuracy compared to Task 1
2. Better discrimination between cancer subtypes due to:
   - Additional methylation features
   - Mutation-methylation interaction patterns
3. Reduced misclassification rates demonstrate the value of multi-omic integration

The confusion matrix is saved as `out/confusion_matrix_task2.png`.